# Ch 11. 파라미터·메모리 계산

10M 모델의 파라미터 수를 config에서 손으로 분해하고,
학습/추론 메모리를 항별로 계산한다.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/desty/study-tiny-llm/blob/main/notebooks/part3/ch11_param_memory.ipynb)

In [ ]:
# 필요한 경우 설치
# !pip install torch

import torch

device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f'device: {device}')
print(f'torch version: {torch.__version__}')

## 최소 예제 1 — 파라미터 수 분해

GPTMini 파라미터를 항별로:

| 항 | 공식 | 설명 |
|---|---|---|
| Embedding | `vocab × D` | weight tying 으로 lm_head 공유 |
| Attention (한 layer) | `4 × D²` | qkv: 3D² + proj: D² |
| FFN SwiGLU (한 layer) | `8 × D²` | w1+w3: 2×D×H, w2: H×D, H≈2.67D |
| Norm (한 layer) | `2D` | 두 RMSNorm의 gamma |
| **한 layer 합계** | **≈ 12 × D²** | |

$$N \approx V \cdot D + L \cdot 12 D^2 + D$$

In [ ]:
def param_count(vocab=8000, n_layer=6, d_model=256, tied=True):
    embed = vocab * d_model
    per_layer = 12 * d_model ** 2          # attn 4D² + FFN 8D²
    layers = n_layer * per_layer
    norm = d_model                          # final RMSNorm
    head = 0 if tied else vocab * d_model
    return embed + layers + norm + head

print(f"{'n_layer':>8} {'d_model':>8} {'params (M)':>12}")
print("-" * 32)
for L, D in [(6, 256), (6, 320), (8, 384), (12, 512), (12, 768)]:
    n = param_count(8000, L, D)
    print(f"  L={L}, D={D:4d}  →  {n / 1e6:6.2f} M")

In [ ]:
# 본 책 기준 (V=8000, L=6, D=320) 항별 분해
V, L, D = 8000, 6, 320

embed = V * D
per_layer_attn = 4 * D ** 2
H = int(8 * D / 3)
H = ((H + 7) // 8) * 8  # 8의 배수
per_layer_ffn = 3 * D * H  # w1 + w2 + w3
per_layer_norm = 2 * D
per_layer = per_layer_attn + per_layer_ffn + per_layer_norm
total = embed + L * per_layer + D

print(f"embed:      {embed:>12,}  ({embed/1e6:.3f}M)")
print(f"attn/layer: {per_layer_attn:>12,}  × {L} = {per_layer_attn*L/1e6:.3f}M")
print(f"ffn/layer:  {per_layer_ffn:>12,}  × {L} = {per_layer_ffn*L/1e6:.3f}M  (H={H})")
print(f"norm/layer: {per_layer_norm:>12,}  × {L} = {per_layer_norm*L/1e6:.3f}M")
print("-" * 45)
print(f"total:      {total:>12,}  ({total/1e6:.2f}M)")

## 최소 예제 2 — 학습 메모리 항별 산수

학습 시 한 step의 메모리:

| 항 | bytes/param | 설명 |
|---|---:|---|
| params (bf16) | 2 | 모델 가중치 |
| grads (bf16) | 2 | 각 파라미터의 gradient |
| Adam m (fp32) | 4 | 1차 moment |
| Adam v (fp32) | 4 | 2차 moment |
| **합계** | **12** | bf16 mixed precision |

activation: `B × T × D × L × c × 2` bytes (c ≈ 14)

In [ ]:
def train_mem_gb(N, B, T, D, L, dtype='bf16', checkpoint=False):
    bpp = 14                                        # bf16 mixed: 14 bytes/param
    param_mem = N * bpp / 1e9
    c_act = 14
    act_mem = B * T * D * L * c_act * 2 / 1e9       # fp16 activation
    if checkpoint:
        act_mem = act_mem / (L ** 0.5)
    return param_mem + act_mem

print(f"10M,  B=32, T=512:  {train_mem_gb(1e7, 32, 512, 320, 6):.2f} GB")
print(f"30M,  B=32, T=512:  {train_mem_gb(3e7, 32, 512, 384, 8):.2f} GB")
print(f"125M, B=8,  T=1024: {train_mem_gb(1.25e8, 8, 1024, 512, 12):.2f} GB")

In [ ]:
# gradient checkpointing 효과 비교
N, B, T, D, L = 1e7, 32, 512, 320, 6

mem_normal = train_mem_gb(N, B, T, D, L, checkpoint=False)
mem_ckpt   = train_mem_gb(N, B, T, D, L, checkpoint=True)

print(f"Without checkpointing: {mem_normal:.2f} GB")
print(f"With checkpointing:    {mem_ckpt:.2f} GB")
print(f"Saving: {mem_normal - mem_ckpt:.2f} GB ({(1 - mem_ckpt/mem_normal)*100:.0f}%)")

## 실전 — 추론 메모리 · KV cache 계산

$$\text{KV cache} = 2 \cdot L \cdot H \cdot d_h \cdot T \cdot \text{bytes}$$

추론 시는 params + KV cache 만. 10M 모델은 KV cache가 작지만,
대형 모델 (7B+) 에서는 KV cache가 모델 크기와 맞먹는다.

In [ ]:
def kv_cache_gb(L, H_kv, d_h, T, bytes_per=2):
    return 2 * L * H_kv * d_h * T * bytes_per / 1e9

# 본 책 10M (L=6, H=8, d_h=40, T=1024, fp16)
print("10M  T=1024:", round(kv_cache_gb(6, 8, 40, 1024) * 1000, 2), "MB")

# Llama 3 8B GQA-8 vs MHA-32
print("Llama 3 8B GQA-8 T=4K:", round(kv_cache_gb(32, 8, 128, 4096), 3), "GB")
print("Llama 3 8B MHA-32 T=4K:", round(kv_cache_gb(32, 32, 128, 4096), 3), "GB")

In [ ]:
# 종합 요약 — 10M 모델 전체 메모리 표
import math

N = param_count(8000, 6, 320)  # 약 10M
print(f"=== 10M 모델 메모리 요약 (N={N/1e6:.2f}M) ===")
print()
print("[학습 시]")
param_bytes = N * 14 / 1e9
act_bytes = 32 * 512 * 320 * 6 * 14 * 2 / 1e9
print(f"  params/grads/Adam: {param_bytes:.2f} GB  (14 bytes/param, bf16 mixed)")
print(f"  activation B=32 T=512: {act_bytes:.2f} GB")
print(f"  합계: {param_bytes + act_bytes:.2f} GB")
print()
print("[추론 시]")
infer_params = N * 2 / 1e9  # bf16
infer_kv = kv_cache_gb(6, 8, 40, 1024)
print(f"  params (bf16): {infer_params*1000:.1f} MB")
print(f"  KV cache T=1024: {infer_kv*1000:.2f} MB")
print(f"  합계: {(infer_params + infer_kv)*1000:.1f} MB")

## 연습문제

1. 본 책 기준 (V=8000, L=6, D=320) 의 파라미터 수를 손으로 계산하고 `param_count()` 와 일치하는지 확인.
2. 같은 10M 인데 (L=2, D=560) 와 (L=12, D=180) 두 config 를 만들어 학습 메모리를 비교. 어느 쪽이 무거운가?
3. activation 메모리 식의 `c` 를 본인 nanoGPT 코드에서 직접 세보라. forward pass 에서 저장되는 중간 텐서 갯수를 기준으로.
4. Llama 3 8B GQA-8 의 KV cache 를 batch=4, T=8K 로 계산. 단일 A100 80GB 에서 동시 사용자 몇 명?
5. **(생각해볼 것)** 같은 파라미터 N 이라면 (deep & thin) vs (shallow & wide) 중 어느 쪽이 메모리 효율적인가? activation 수식 관점에서.

In [ ]:
# 연습문제 1: 손으로 계산 후 param_count() 와 비교
V, L, D = 8000, 6, 320

# 직접 계산
my_embed = V * D
my_attn = L * 4 * D**2
H_swiglu = int(8 * D / 3); H_swiglu = ((H_swiglu + 7) // 8) * 8
my_ffn = L * 3 * D * H_swiglu
my_norm = L * 2 * D + D  # per-block norms + final
my_total = my_embed + my_attn + my_ffn + my_norm

auto_total = param_count(V, L, D)

print(f"손 계산: {my_total/1e6:.3f}M")
print(f'param_count(): {auto_total/1e6:.3f}M')
print(f"차이: {abs(my_total - auto_total):,}")

In [ ]:
# 연습문제 2: (L=2, D=560) vs (L=12, D=180) 학습 메모리 비교
# 두 config의 파라미터 수가 비슷한지 확인 후 메모리 비교


In [ ]:
# 연습문제 3: nanoGPT forward 에서 중간 텐서 c 세기
# torch.autograd.graph 나 직접 주석으로 세어보기


In [ ]:
# 연습문제 4: Llama 3 8B GQA-8, batch=4, T=8K KV cache 계산
kv_per_user = kv_cache_gb(32, 8, 128, 8192)  # 1명
model_size_gb = 8e9 * 2 / 1e9               # 8B params, bf16
a100_gb = 80

available = a100_gb - model_size_gb
max_users = int(available / kv_per_user)

print(f"모델 크기: {model_size_gb:.0f} GB")
print(f"사용자당 KV cache: {kv_per_user*1000:.0f} MB")
print(f"A100 80GB 에서 가용: {available:.0f} GB")
print(f"최대 동시 사용자 (batch=1 per user): {max_users} 명")
print(f"최대 동시 사용자 (batch=4): {max_users // 4} 명")

In [ ]:
# 연습문제 5: deep & thin vs shallow & wide 메모리 비교
# activation = B * T * D * L * c * 2
B, T, c = 32, 512, 14

# 같은 N ≈ 10M 을 두 가지로
configs_cmp = [
    (12, 180, "deep & thin"),
    (2,  560, "shallow & wide"),
]

print(f"{'config':>20} {'N (M)':>8} {'act (GB)':>10} {'total (GB)':>12}")
print("-" * 55)
for L, D, name in configs_cmp:
    N = param_count(8000, L, D)
    act = B * T * D * L * c * 2 / 1e9
    total = N * 14 / 1e9 + act
    print(f"{name:>20} {N/1e6:>8.2f} {act:>10.3f} {total:>12.3f}")